In [ ]:
import numpy as np
import pandas as pd
from collections import deque
from provenance import Provenance
import algebra

t = Provenance()

In [ ]:
# --- Data Loading ---
print("LOADING REAL BIBLE DATA (BradyStephenson/bible-data)")

base_url = "https://raw.githubusercontent.com/BradyStephenson/bible-data/main"
df_persons = pd.read_csv(f"{base_url}/BibleData-Person.csv")
df_rels    = pd.read_csv(f"{base_url}/BibleData-PersonRelationship.csv")

print(f"Loaded {len(df_persons)} people and {len(df_rels)} relationship records.")

In [ ]:
# --- Normalize to Parent → Child direction ---
target_rels = ['father', 'mother', 'son', 'daughter']
df_family = df_rels[df_rels['relationship_type'].str.lower().isin(target_rels)].copy()

mask_parent = df_family['relationship_type'].str.lower().isin(['father', 'mother'])

df_family['parent_id'] = np.where(mask_parent, df_family['person_id_1'], df_family['person_id_2'])
df_family['child_id']  = np.where(mask_parent, df_family['person_id_2'], df_family['person_id_1'])
df_family = df_family.dropna(subset=['parent_id', 'child_id'])

In [ ]:
# --- Build adjacency matrix ---
valid_ids = sorted(set(df_family['parent_id']) | set(df_family['child_id']))
id_to_idx = {rid: i for i, rid in enumerate(valid_ids)}
N         = len(valid_ids)

id_to_name = df_persons.set_index('person_id')['person_name']
names      = [id_to_name.get(rid, f"Unknown_{rid}") for rid in valid_ids]

Parent = np.zeros((N, N), dtype=int)
p_idx  = [id_to_idx[p] for p in df_family['parent_id']]
c_idx  = [id_to_idx[c] for c in df_family['child_id']]
Parent[p_idx, c_idx] = 1

print(f"Matrix: {Parent.shape} ({N} nodes)")

In [ ]:
def print_relationships(matrix, title):
    count = int(np.sum(matrix))
    print(f"\n{title}: {count} relationships found")
    return count

print_relationships(Parent, "Direct parent relationships")

In [ ]:
print("\nCONVERGENCE ITERATIONS")
n = Parent.shape[0]
Edge = np.random.uniform(0, 0.1, (n, n))
family_tree = t.Closure(Parent, temp=1.0)

In [ ]:
# --- Verify ---
def verify_transitive_closure(P, A):
    P = (P > 0).astype(int)
    A = (A > 0).astype(int)

    print("[1/3] Checking: Parent ⊆ Ancestor")
    assert np.all((P == 0) | (A == 1)), "Missing a direct parent link in Ancestor"
    print("  ✓ PASS")

    print("[2/3] Checking: Closure property (A @ Parent adds nothing)")
    assert not np.any((A @ P > 0) & (A == 0)), "Closure not reached (A @ Parent found new edges)"
    print("  ✓ PASS")

    print("[3/3] Checking: No cycles (diagonal = 0)")
    cycles = int(np.diag(A).sum())
    if cycles > 0:
        print(f"  ! NOTE: {cycles} self-loops found (data quirks / loops in raw dataset).")
    else:
        print("  ✓ PASS")

print("\nFINAL RESULTS")
initial_count = print_relationships(Parent,      "Direct parent relationships")
final_count   = print_relationships(family_tree, "Complete ancestor relationships")
print(f"Total new links discovered: {final_count - initial_count}")


In [ ]:
try:
    verify_transitive_closure(Parent, family_tree)
    print("\n *** ALL CHECKS PASSED ***")
except AssertionError as e:
    print(f"\n --- Verification failed --- : {e}")

In [ ]:
# --- Lineage queries ---
def show_lineage(name_query, top_n=None):
    def bfs_levels(adj_matrix, start_idx):
        dist = {start_idx: 0}
        q = deque([start_idx])
        while q:
            u = q.popleft()
            for v in np.where(adj_matrix[u] == 1)[0]:
                if v not in dist:
                    dist[v] = dist[u] + 1
                    q.append(v)
        return dist

    try:
        idx = next(i for i, n in enumerate(names) if name_query.lower() in n.lower())
    except StopIteration:
        print(f"\nCould not find '{name_query}' in the dataset.")
        return

    name       = names[idx]
    total_desc = int((family_tree[idx] > 0).sum())
    total_anc  = int((family_tree[:, idx] > 0).sum())

    print(f"\n[{name}]")
    print(f"  → Ancestor of {total_desc} people in total.")
    print(f"  ← Descended from {total_anc} people in total.")

    if top_n is None:
        return

    desc_dist = bfs_levels(Parent, idx)
    desc_dist.pop(idx, None)
    sorted_desc = sorted(desc_dist.items(), key=lambda x: (x[1], names[x[0]]))[:top_n]

    anc_dist = bfs_levels(Parent.T, idx)
    anc_dist.pop(idx, None)
    sorted_anc = sorted(anc_dist.items(), key=lambda x: (x[1], names[x[0]]))[:top_n]

    if sorted_desc:
        print(f"Top {len(sorted_desc)} descendants by generation:")
        for node_idx, dist in sorted_desc:
            print(f"    gen {dist:2d}: {names[node_idx]}")
    else:
        print("No descendants in the dataset.")

    if sorted_anc:
        print(f"Top {len(sorted_anc)} ancestors by generation:")
        for node_idx, dist in sorted_anc:
            print(f"    gen {dist:2d}: {names[node_idx]}")
    else:
        print("No ancestors in the dataset.")

In [ ]:
print("\nINTERPRETATION (Selected Bible Figures)")
show_lineage("Adam")
show_lineage("Adam", top_n=5)
show_lineage("Abram", top_n=10)

In [ ]:

# Find indices for test figures
adam_idx  = next(i for i, n in enumerate(names) if "Adam"  == n)
isaac_idx = next(i for i, n in enumerate(names) if "Isaac" == n)
abram_idx = next(i for i, n in enumerate(names) if "Abram" == n)

print(f"Adam  → idx {adam_idx}  ({names[adam_idx]})")
print(f"Isaac → idx {isaac_idx} ({names[isaac_idx]})")
print(f"Abram → idx {abram_idx} ({names[abram_idx]})")

# Query: is Adam an ancestor of Abram?
result = t.Query(family_tree, src=adam_idx, dst=abram_idx, names=names, relation="ancestor of")
print("\n--- Adam → Abram ---")
print(result)